In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from catboost import Pool, CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

In [2]:
train_extra = pl.read_parquet('data/train_extra_features.parquet')
test_extra = pl.read_parquet('data/test_extra_features.parquet')

target = pl.read_parquet('data/train_target.parquet')

print('Тренировочные данные (extra):', train_extra.shape)
print('Тестовые данные (extra):', test_extra.shape)

Тренировочные данные (extra): (750000, 2242)
Тестовые данные (extra): (250000, 2242)


In [3]:
train_extra.head(n = 5)

customer_id,num_feature_133,num_feature_134,num_feature_135,num_feature_136,num_feature_137,num_feature_138,num_feature_139,num_feature_140,num_feature_141,num_feature_142,num_feature_143,num_feature_144,num_feature_145,num_feature_146,num_feature_147,num_feature_148,num_feature_149,num_feature_150,num_feature_151,num_feature_152,num_feature_153,num_feature_154,num_feature_155,num_feature_156,num_feature_157,num_feature_158,num_feature_159,num_feature_160,num_feature_161,num_feature_162,num_feature_163,num_feature_164,num_feature_165,num_feature_166,num_feature_167,num_feature_168,…,num_feature_2337,num_feature_2338,num_feature_2339,num_feature_2340,num_feature_2341,num_feature_2342,num_feature_2343,num_feature_2344,num_feature_2345,num_feature_2346,num_feature_2347,num_feature_2348,num_feature_2349,num_feature_2350,num_feature_2351,num_feature_2352,num_feature_2353,num_feature_2354,num_feature_2355,num_feature_2356,num_feature_2357,num_feature_2358,num_feature_2359,num_feature_2360,num_feature_2361,num_feature_2362,num_feature_2363,num_feature_2364,num_feature_2365,num_feature_2366,num_feature_2367,num_feature_2368,num_feature_2369,num_feature_2370,num_feature_2371,num_feature_2372,num_feature_2373
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,-0.116197,-0.009981,null,null,null,null,-0.067005,null,null,-0.001311,0.0,null,-0.087608,-0.011702,null,null,null,null,null,null,null,null,null,-0.024235,null,-0.049388,-0.034772,null,-0.038661,-0.075484,1.116927,-0.032538,-0.122623,-0.01873,-0.081526,null,…,-0.039189,null,0.0,null,null,-0.14431,null,null,-0.053982,null,0.0,-0.072911,null,-0.581945,null,-0.080336,0.440102,null,null,-0.020938,-0.142138,null,-0.023246,0.0,null,-0.020769,-0.14374,null,-0.00276,null,0.0,-0.030607,-0.015407,-0.404816,-0.395589,null,0.0
1000002,-0.116197,-0.009981,null,null,null,null,-0.067005,-0.088452,null,0.000073,0.0,null,-0.087608,-0.011702,-0.280531,null,-0.001478,null,null,null,-0.15721,null,null,-0.024235,0.550582,-0.049388,-0.034772,null,-0.038661,-0.075484,-0.232695,-0.032538,-0.122623,0.176785,-0.081526,null,…,-0.023996,null,0.0,null,null,-0.14431,null,null,-0.053982,2.186477,0.0,-0.072911,null,0.04981,-0.035824,-0.080336,0.186329,null,null,-0.020938,-0.122687,2.700091,-0.023246,0.0,null,-0.020769,-0.14374,null,-0.001345,null,0.0,-0.030607,-0.047964,1.630249,-0.395589,-0.201813,0.0
1000003,-0.116197,-0.009981,null,null,null,null,-0.067005,-0.088452,null,-0.001343,0.0,null,-0.087608,-0.011702,3.639459,null,-0.001488,null,null,null,null,null,null,-0.024235,-0.325277,-0.049388,-0.034772,null,null,-0.075484,-0.232695,-0.032538,-0.122623,-0.01873,-0.081526,null,…,-0.039189,null,0.0,null,null,-0.14431,null,null,-0.053982,-0.140279,0.0,-0.072911,null,-0.660424,-0.035824,-0.080336,-0.942263,null,null,null,-0.228213,-0.143084,-0.023246,0.0,null,-0.020769,-0.14374,null,-0.00276,null,0.0,-0.030607,-0.045637,-0.404816,-0.395589,-0.271623,0.0
1000004,-0.116197,-0.009981,null,null,null,null,-0.067005,null,null,-0.001272,0.0,-1.18569,-0.087608,-0.011702,1.679464,null,null,-0.898284,null,null,null,null,null,-0.024235,null,-0.049388,-0.034772,null,null,-0.075484,-0.232695,null,-0.122623,null,-0.081526,null,…,null,null,0.0,-0.318906,null,-0.14431,null,null,-0.053982,null,0.0,-0.072911,null,-0.683968,null,-0.080336,0.528574,-0.657523,null,null,-0.352937,null,-0.023246,0.0,0.0,-0.020769,-0.14374,null,-0.00276,null,0.0,-0.030607,-0.058314,-0.404816,null,null,0.0
1000005,null,null,null,null,null,null,null,-0.088452,null,null,0.0,null,null,-0.011702,0.046135,null,-0.001492,null,null,null,null,null,null,-0.024235,-0.150105,null,null,null,null,-0.075484,null,-0.032538,null,-0.01873,null,null,…,-0.039189,null,0.0,null,null,null,null,null,null,-0.140279,0.0,-0.072911,nu

In [7]:
import polars as pl
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from tqdm.auto import tqdm

# --- признаки и таргет ---
extra_feature_cols = [col for col in train_extra.columns if col != "customer_id"]
target_columns = [col for col in target.columns if col.startswith("target")]

X_extra_pl = train_extra.select(extra_feature_cols)

# ----------------------------
# 1) индикаторы пропусков
# ----------------------------
missing_indicators = (
    X_extra_pl
    .select(pl.all().is_null().cast(pl.Int8))
    .rename({c: f"{c}_isnull" for c in X_extra_pl.columns})
)

# ----------------------------
# 2) разделяем по типам
# ----------------------------
schema = X_extra_pl.schema
num_cols = [c for c, dt in schema.items() if dt.is_numeric()]
str_cols = [c for c, dt in schema.items() if dt in (pl.Utf8, pl.Categorical, pl.Enum)]

# ----------------------------
# 3) медианы для числовых
# ----------------------------
medians_row = (
    X_extra_pl
    .select([pl.col(c).median().alias(c) for c in num_cols])
    .row(0, named=True)
)

# ----------------------------
# 4) заполнение пропусков
# ----------------------------
X_extra_filled = X_extra_pl.with_columns(
    [pl.col(c).fill_null(medians_row[c]) for c in num_cols] +
    [pl.col(c).fill_null("missing") for c in str_cols]
)

# ----------------------------
# 5) объединяем признаки
# ----------------------------
X_extra_full = pl.concat(
    [X_extra_filled, missing_indicators],
    how="horizontal"
)

# если есть строковые — кодируем в категории → числа
if str_cols:
    X_extra_full = X_extra_full.with_columns(
        [pl.col(c).cast(pl.Categorical).to_physical() for c in str_cols]
    )

# в pandas
X_extra = X_extra_full.to_pandas()

# ----------------------------
# target
# ----------------------------
y_extra = target.select(target_columns).to_pandas()

# если один таргет → делаем 1D
if y_extra.shape[1] == 1:
    y_extra = y_extra.values.ravel()
else:
    y_extra = y_extra.values  # multioutput


# ============================
# ExtraTrees с прогрессом
# ============================

n_estimators_total = 300
step = 10

extra_selector = ExtraTreesClassifier(
    n_estimators=step,
    warm_start=True,              # ВАЖНО
    max_depth=10,
    min_samples_split=50,
    min_samples_leaf=20,
    max_features="sqrt",
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

for n in tqdm(range(step, n_estimators_total + 1, step),
              desc="Training ExtraTrees"):
    extra_selector.set_params(n_estimators=n)
    extra_selector.fit(X_extra, y_extra)


# ============================
# Важности признаков
# ============================

all_features = X_extra.columns

extra_importance = (
    pl.DataFrame({
        "feature": all_features,
        "importance": extra_selector.feature_importances_
    })
    .sort("importance", descending=True)
)

selected_extra_features = (
    extra_importance
    .head(100)["feature"]
    .to_list()
)

Training ExtraTrees:   0%|          | 0/30 [00:00<?, ?it/s]

/home/test/miniconda3/envs/halls_cola/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:860: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
/home/test/miniconda3/envs/halls_cola/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:860: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequ

FileNotFoundError: No such file or directory (os error 2): modified_data/selected_extra_features.parquet

In [8]:
from pathlib import Path

output_dir = Path("modified_data")
output_dir.mkdir(parents=True, exist_ok=True)

pl.DataFrame({"feature": selected_extra_features}).write_parquet(
    output_dir / "selected_extra_features.parquet"
)

extra_importance.write_parquet(
    output_dir / "extra_importance.parquet"
)

In [13]:
import polars as pl

# --- загрузка сохранённого списка ---
selected_extra_features = (pl.read_parquet("modified_data/selected_extra_features.parquet")["feature"].to_list())

print("Выбрано extra-признаков:", len(selected_extra_features))
selected_extra_features

Выбрано extra-признаков: 100


['num_feature_1775_isnull',
 'num_feature_925_isnull',
 'num_feature_1478_isnull',
 'num_feature_2343_isnull',
 'num_feature_319_isnull',
 'num_feature_395_isnull',
 'num_feature_809_isnull',
 'num_feature_376_isnull',
 'num_feature_1567_isnull',
 'num_feature_1377_isnull',
 'num_feature_1982_isnull',
 'num_feature_2200_isnull',
 'num_feature_390_isnull',
 'num_feature_613_isnull',
 'num_feature_201_isnull',
 'num_feature_751_isnull',
 'num_feature_2316_isnull',
 'num_feature_1761_isnull',
 'num_feature_1287',
 'num_feature_1851_isnull',
 'num_feature_650',
 'num_feature_176_isnull',
 'num_feature_2299_isnull',
 'num_feature_1902_isnull',
 'num_feature_1109_isnull',
 'num_feature_1394_isnull',
 'num_feature_1218_isnull',
 'num_feature_2059_isnull',
 'num_feature_1173_isnull',
 'num_feature_1139_isnull',
 'num_feature_1545_isnull',
 'num_feature_466_isnull',
 'num_feature_450_isnull',
 'num_feature_820_isnull',
 'num_feature_1693_isnull',
 'num_feature_345_isnull',
 'num_feature_719_isn

In [10]:
train = train.join(
    train_extra.select(["customer_id"] + selected_extra_features),
    on="customer_id",
    how="left"
)
test = test.join(
    test_extra.select(["customer_id"] + selected_extra_features),
    on="customer_id",
    how="left"
)


print("train shape после join:", train.shape)
print("test shape после join:", test.shape)

NameError: name 'train' is not defined